# RL Stage — Adaptive Permission Risk Weights

This notebook adds a Reinforcement Learning stage on top of the existing
Borda + Garrett permission-risk pipeline (`dummy_permissionMatrix.ipynb`).

**Design summary**

- **Problem framing:** one episode = the agent walks through the 12
  permissions in sequence and decides whether to nudge each permission's
  risk weight up, down, or leave it unchanged. This gives genuine
  sequential structure (later permissions are adjusted *given* the
  weights already set and re-normalized), which is what justifies calling
  it an MDP rather than a one-shot optimization dressed up as RL.
- **State:** `(permission_index, discretized_weight_bin)`
- **Action space:** `{-1 (decrease), 0 (keep), +1 (increase)}` applied to
  the current permission's weight, by a fixed step size, followed by
  renormalization so weights always sum to 1.
- **Reward:** shaped reward = change in F1 (computed with **NoisyLabel**,
  never TrueLabel) after the step, minus a risk-consistency penalty that
  discourages the agent from chasing noisy labels too aggressively. A
  terminal bonus is added at the end of the episode based on final F1 +
  AUC on the training fold only.
- **Environment:** wraps the **training fold only**. Each episode resets
  weights to the Borda+Garrett baseline before the agent starts adjusting.
- **Agent objective:** maximize cumulative reward → learn a correction to
  the Garrett weights that improves noisy-label separability without
  drifting arbitrarily far from the baseline.
- **Baseline interaction:** RL is a *warm-started correction*, not a
  from-scratch learner. Final weight = `baseline_weight + learned_delta`,
  clipped and renormalized.
- **What RL changes:** only the permission weights. Thresholds and the
  classification rule are held fixed so the ablation (Borda vs Garrett vs
  Borda+Garrett vs +RL) isolates the effect of weight adaptation.
- **Leakage control:** RL never sees the test fold or TrueLabel during
  training. TrueLabel is used only in the final evaluation cell and in the
  label-flip diagnostic at the end.
- **Algorithm:** tabular Q-learning. With 12 permissions × a handful of
  weight bins × 3 actions, the state-action space is small and fully
  enumerable — a deep method (DQN/PPO) would add variance and
  hyperparameter surface without any evidence it's needed at n=200.


## 1. Setup & Load Data

Assumes this notebook sits alongside `android_apps_dataset_social.csv`, `social_permission_matrix.csv`, and `permission_risk_weights.csv` in the same `PrivacyScoreAndriodApps` folder.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import random

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

PERMISSIONS = [
    "Camera", "Location", "Contacts", "Microphone", "Storage", "SMS",
    "CallLogs", "Bluetooth", "Calendar", "PhoneState", "Internet",
    "ReadPhoneNumbers"
]

df = pd.read_csv("android_apps_dataset_social.csv")
print(df.shape)
df.head()

In [ ]:
# Sanity checks on the columns the RL stage depends on
required_cols = PERMISSIONS + ["TrueLabel", "NoisyLabel", "PrivacyRiskScore"]
missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing expected columns: {missing}"

if "LabelFlipped" not in df.columns:
    print("Warning: LabelFlipped column not found — the flip-detection diagnostic at the end will be skipped.")

df[PERMISSIONS] = df[PERMISSIONS].astype(int)
df["TrueLabel"] = df["TrueLabel"].astype(int)
df["NoisyLabel"] = df["NoisyLabel"].astype(int)
df["NoisyLabel"].value_counts()

## 2. Train / Test Split (leakage-safe, same convention as the baseline notebook)

Stratify on `TrueLabel` for a fair test set, but the RL agent will only ever be given `NoisyLabel` and the training rows.

In [ ]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df["TrueLabel"], random_state=RANDOM_STATE
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("Train:", train_df.shape, " Test:", test_df.shape)
print("Train TrueLabel balance:\n", train_df["TrueLabel"].value_counts(normalize=True))

## 3. Baseline (Garrett) Weights — loaded from your existing pipeline

If `permission_risk_weights.csv` already holds train-fold-only weights, load it directly. Otherwise this recomputes Garrett weights from the training fold only, so nothing here reuses weights derived from the full dataset.

In [ ]:
def garrett_percentile(rank, n):
    return 100.0 * (rank - 0.5) / n

def compute_garrett_weights(frame, permissions):
    n = len(permissions)
    freq = frame[permissions].sum().rename("Frequency")
    risk = pd.Series(
        {p: frame.loc[frame[p] == 1, "PrivacyRiskScore"].mean() for p in permissions},
        name="AvgRisk"
    )
    mal_assoc = pd.Series(
        {p: frame.loc[frame[p] == 1, "NoisyLabel"].mean() for p in permissions},
        name="PMalicious"
    )
    ranks = pd.DataFrame({
        "Frequency": freq.rank(ascending=False, method="average"),
        "AvgRisk": risk.rank(ascending=False, method="average"),
        "PMalicious": mal_assoc.rank(ascending=False, method="average"),
    })
    garrett_table = {r: garrett_percentile(r, n) for r in range(1, n + 1)}
    def to_garrett(rank_series):
        return rank_series.apply(lambda r: np.interp(r, list(garrett_table.keys()), list(garrett_table.values())))
    garrett_scores = ranks.apply(to_garrett)
    garrett_total = garrett_scores.sum(axis=1)
    weights = garrett_total / garrett_total.sum()
    return weights.reindex(permissions)

try:
    baseline_weights = pd.read_csv("permission_risk_weights.csv", index_col=0).iloc[:, 0]
    baseline_weights = baseline_weights.reindex(PERMISSIONS)
    if baseline_weights.isna().any():
        raise ValueError("Loaded weights don't cover all 12 permissions — recomputing from train fold.")
    print("Loaded baseline weights from permission_risk_weights.csv")
except Exception as e:
    print("Falling back to recomputing Garrett weights from the training fold:", e)
    baseline_weights = compute_garrett_weights(train_df, PERMISSIONS)

baseline_weights = baseline_weights / baseline_weights.sum()
baseline_weights.sort_values(ascending=False)

## 4. Environment

One episode = one full pass over the 12 permissions. At each step the agent nudges the current permission's weight, weights are renormalized, and a shaped reward is returned. A terminal bonus is added on the final step.

In [ ]:
WEIGHT_STEP = 0.01          # how much one action changes a weight before renormalization
N_BINS = 11                 # discretization bins for the state (0.00 .. 0.10+ in steps of ~0.01)
BIN_EDGES = np.linspace(0, 0.30, N_BINS)  # weights above this range collapse into the last bin
CONSISTENCY_LAMBDA = 0.15    # penalty strength for drifting from baseline behavior

def discretize(w):
    return int(np.clip(np.digitize(w, BIN_EDGES) - 1, 0, N_BINS - 1))

def compute_scores(weights, frame):
    w = weights.reindex(PERMISSIONS).values
    perm_matrix = frame[PERMISSIONS].values
    return perm_matrix @ w  # BaselinePrivacyScore-style score in [0,1]

def scores_to_preds(scores, threshold=0.5):
    return (scores >= threshold).astype(int)

def f1_on(weights, frame, threshold=0.5):
    scores = compute_scores(weights, frame)
    preds = scores_to_preds(scores, threshold)
    return f1_score(frame["NoisyLabel"], preds, zero_division=0), scores

class PermissionWeightEnv:
    """
    Episode = one pass through the 12 permissions.
    State   = (permission_index, discretized_weight_bin_for_that_permission)
    Action  = -1 / 0 / +1 applied to that permission's weight
    Reward  = shaped F1 delta (NoisyLabel, train fold only) minus a
              consistency penalty, with a terminal bonus.
    """
    def __init__(self, train_frame, base_weights, permissions=PERMISSIONS):
        self.train_frame = train_frame
        self.base_weights = base_weights.copy()
        self.permissions = permissions
        self.n_perm = len(permissions)

    def reset(self):
        self.weights = self.base_weights.copy()
        self.step_idx = 0
        self.prev_f1, _ = f1_on(self.weights, self.train_frame)
        return self._state()

    def _state(self):
        p = self.permissions[self.step_idx]
        return (self.step_idx, discretize(self.weights[p]))

    def step(self, action):
        p = self.permissions[self.step_idx]
        delta = {-1: -WEIGHT_STEP, 0: 0.0, 1: WEIGHT_STEP}[action]
        self.weights[p] = max(0.0, self.weights[p] + delta)
        self.weights = self.weights / self.weights.sum()

        new_f1, scores = f1_on(self.weights, self.train_frame)
        drift_penalty = CONSISTENCY_LAMBDA * np.abs(
            self.weights.values - self.base_weights.values
        ).sum() / self.n_perm

        reward = (new_f1 - self.prev_f1) - drift_penalty
        self.prev_f1 = new_f1

        self.step_idx += 1
        done = self.step_idx >= self.n_perm

        if done:
            auc = roc_auc_score(self.train_frame["NoisyLabel"], scores) if len(set(self.train_frame["NoisyLabel"])) > 1 else 0.5
            reward += new_f1 + auc  # terminal bonus
            next_state = None
        else:
            next_state = self._state()

        return next_state, reward, done, {"f1": new_f1}


## 5. Tabular Q-Learning Agent

In [ ]:
class QLearningAgent:
    def __init__(self, n_states_per_step, n_perm, n_actions=3,
                 alpha=0.1, gamma=0.95, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.05):
        # Q-table keyed by (step_idx, weight_bin, action)
        self.Q = np.zeros((n_perm, n_states_per_step, n_actions))
        self.actions = [-1, 0, 1]
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def act(self, state):
        step_idx, bin_idx = state
        if random.random() < self.epsilon:
            return random.choice(range(len(self.actions)))
        return int(np.argmax(self.Q[step_idx, bin_idx]))

    def update(self, state, action_idx, reward, next_state, done):
        step_idx, bin_idx = state
        current_q = self.Q[step_idx, bin_idx, action_idx]
        if done or next_state is None:
            target = reward
        else:
            n_step, n_bin = next_state
            target = reward + self.gamma * np.max(self.Q[n_step, n_bin])
        self.Q[step_idx, bin_idx, action_idx] += self.alpha * (target - current_q)

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


## 6. Training Loop

In [ ]:
env = PermissionWeightEnv(train_df, baseline_weights)
agent = QLearningAgent(n_states_per_step=N_BINS, n_perm=len(PERMISSIONS))

N_EPISODES = 800
reward_history = []

for ep in range(N_EPISODES):
    state = env.reset()
    total_reward = 0.0
    done = False
    while not done:
        action_idx = agent.act(state)
        action = agent.actions[action_idx]
        next_state, reward, done, info = env.step(action)
        agent.update(state, action_idx, reward, next_state, done)
        state = next_state
        total_reward += reward
    agent.decay_epsilon()
    reward_history.append(total_reward)

print("Final epsilon:", round(agent.epsilon, 3))
print("Mean reward, last 50 episodes:", round(np.mean(reward_history[-50:]), 4))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(reward_history)
plt.xlabel("Episode")
plt.ylabel("Total episode reward")
plt.title("Q-learning training curve (train fold, NoisyLabel-shaped reward)")
plt.tight_layout()
plt.show()

## 7. Extract Learned Weights

Greedy rollout through the trained Q-table gives the final RL-adjusted weight vector.

In [ ]:
def greedy_rollout(env, agent):
    state = env.reset()
    done = False
    while not done:
        step_idx, bin_idx = state
        action_idx = int(np.argmax(agent.Q[step_idx, bin_idx]))
        action = agent.actions[action_idx]
        state, reward, done, info = env.step(action)
    return env.weights.copy()

rl_weights = greedy_rollout(env, agent)
comparison = pd.DataFrame({
    "Garrett_baseline": baseline_weights,
    "RL_adjusted": rl_weights,
    "Delta": rl_weights - baseline_weights
}).sort_values("Delta", ascending=False)
comparison

## 8. Final Evaluation on the Held-Out Test Fold (TrueLabel, never seen by the agent)

In [ ]:
def evaluate(weights, frame, label_col="TrueLabel", threshold=0.5):
    scores = compute_scores(weights, frame)
    preds = scores_to_preds(scores, threshold)
    y = frame[label_col]
    return {
        "Accuracy": accuracy_score(y, preds),
        "Precision": precision_score(y, preds, zero_division=0),
        "Recall": recall_score(y, preds, zero_division=0),
        "F1": f1_score(y, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y, scores) if len(set(y)) > 1 else float("nan"),
    }, scores

results_rl, test_scores_rl = evaluate(rl_weights, test_df, "TrueLabel")
results_baseline, test_scores_baseline = evaluate(baseline_weights, test_df, "TrueLabel")

pd.DataFrame({"Garrett_baseline": results_baseline, "Borda_Garrett_RL": results_rl})

## 9. Full Method Comparison

Frequency-only and Borda-only weights are computed fresh from the training fold to keep every method on the same leakage-safe footing, then all five methods are scored on the same test fold.

In [ ]:
def frequency_only_weights(frame, permissions):
    freq = frame[permissions].sum()
    return freq / freq.sum()

def borda_only_weights(frame, permissions):
    n = len(permissions)
    freq = frame[permissions].sum()
    risk = pd.Series({p: frame.loc[frame[p] == 1, "PrivacyRiskScore"].mean() for p in permissions})
    mal = pd.Series({p: frame.loc[frame[p] == 1, "NoisyLabel"].mean() for p in permissions})
    ranks = pd.DataFrame({
        "f": freq.rank(ascending=False),
        "r": risk.rank(ascending=False),
        "m": mal.rank(ascending=False),
    })
    borda_points = (n - ranks) + 1  # higher rank -> more points
    borda_total = borda_points.sum(axis=1)
    return (borda_total / borda_total.sum()).reindex(permissions)

freq_w = frequency_only_weights(train_df, PERMISSIONS)
borda_w = borda_only_weights(train_df, PERMISSIONS)
garrett_w = baseline_weights  # already Garrett-based
borda_garrett_w = (borda_w + garrett_w) / 2  # simple ensemble, matches your existing "Borda+Garrett" step
borda_garrett_rl_w = rl_weights

methods = {
    "Frequency-only": freq_w,
    "Borda-only": borda_w,
    "Garrett-only": garrett_w,
    "Borda+Garrett": borda_garrett_w,
    "Borda+Garrett+RL": borda_garrett_rl_w,
}

rows = []
for name, w in methods.items():
    metrics, _ = evaluate(w, test_df, "TrueLabel")
    metrics["Method"] = name
    rows.append(metrics)

comparison_table = pd.DataFrame(rows).set_index("Method")[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]
comparison_table.round(4)

## 10. Label-Flip Detection Diagnostic

Using the RL-adjusted weights, flag any row where the computed risk score sharply disagrees with its `NoisyLabel`. Then check those flags against `LabelFlipped` (ground truth of which rows were deliberately corrupted), computed on the **full dataset** purely as a diagnostic — this is not part of the model's train/test evaluation.

In [ ]:
if "LabelFlipped" in df.columns:
    all_scores = compute_scores(rl_weights, df)
    disagreement = np.abs(all_scores - df["NoisyLabel"])  # score in [0,1] vs {0,1} label
    flag_threshold = disagreement.quantile(0.85)  # top 15% most inconsistent rows, tune as needed
    flagged = (disagreement >= flag_threshold).astype(int)

    flip_precision = precision_score(df["LabelFlipped"], flagged, zero_division=0)
    flip_recall = recall_score(df["LabelFlipped"], flagged, zero_division=0)
    flip_f1 = f1_score(df["LabelFlipped"], flagged, zero_division=0)

    print(f"Flip-detection precision: {flip_precision:.3f}")
    print(f"Flip-detection recall:    {flip_recall:.3f}")
    print(f"Flip-detection F1:        {flip_f1:.3f}")
    print(f"(Threshold flags top {(flagged.mean()*100):.1f}% of rows by score/label disagreement)")
else:
    print("No LabelFlipped column present — skipping this diagnostic.")

## 11. Save Artifacts

In [ ]:
rl_weights.rename("RL_Weight").to_csv("rl_permission_risk_weights.csv")
comparison_table.to_csv("method_comparison_results.csv")
print("Saved: rl_permission_risk_weights.csv, method_comparison_results.csv")

## Notes for the write-up

- The reward function is deliberately shaped on **NoisyLabel**, not TrueLabel — this is what makes the
  label-flip diagnostic in section 10 meaningful. If the agent were rewarded on TrueLabel there would be
  nothing interesting to "recover."
- `CONSISTENCY_LAMBDA`, `WEIGHT_STEP`, and the flip-detection quantile threshold are all hyperparameters
  worth reporting a small sensitivity sweep for in the final report — cite the values you settle on and
  show how F1/AUC and flip-detection recall move across a couple of alternatives.
- If you move to the multi-category dataset, `N_BINS`/`BIN_EDGES` and `N_EPISODES` should be revisited;
  more permissions or more categories will need a larger, more granular state space, at which point the
  case for a function-approximation method (DQN) becomes stronger and worth re-arguing explicitly rather
  than assumed.
